After consulting with ChatGPT and Gemini about what to do with this dataset since having post-game information/stats is not really "predicting" a game, they explained that what is meant to do with a historical table like this in ML is to create a dataset of rolling features that consider the only previous N results in trying to predict the current game which we consider unknown. AKA, we should compute an average historical stat to use in predicting the outcome of the current game.

In [117]:
import pandas as pd
import numpy as np
df = pd.read_csv("../../data/cs2_tier1_games.csv", encoding="latin1")
# Help from ChatGPT to convert to real datetime
df.datetime = pd.to_datetime(df.datetime, format="%Y/%m/%d %H:%M", errors="coerce")
df = df.sort_values(by="datetime")
df

,Unnamed: 0,match_id,game_id,tournament,team1_id,team1,team2_id,team2,score1_match,score2_match,...,team2_player4_assists,team2_player4_adr,team2_player4_kast,team2_player4_kddiff,team2_player5_kills,team2_player5_deaths,team2_player5_assists,team2_player5_adr,team2_player5_kast,team2_player5_kddiff
9071,9071,589049,-115902,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,14.0,60.7,73.2,-9.0,39.0,50.0,12.0,63.9,63.3,-11.0
9068,9068,589049,115902,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,7.0,80.1,83.3,1.0,13.0,16.0,4.0,49.0,58.3,-3.0
9070,9070,589049,115901,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,2.0,48.0,66.7,-4.0,9.0,17.0,6.0,74.0,61.9,-8.0
9069,9069,589049,115903,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,5.0,54.0,69.6,-6.0,17.0,17.0,2.0,68.6,69.6,0.0
9067,9067,589047,-115906,Thunderpick World Championship 2023,233631,Monte,233632,Wildcard Gaming,2,0,...,6.0,56.3,45.0,-15.0,14.0,29.0,2.0,53.7,47.6,-15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,4,7230540,125821,BLAST Open Spring 2026,288894,PARIVISION,288895,Natus Vincere,1,2,...,5.0,85.8,80.0,2.0,23.0,15.0,7.0,94.2,75.0,8.0
3,3,7230540,125819,BLAST Open Spring 2026,288894,PARIVISION,288895,Natus Vincere,1,2,...,6.0,78.9,76.2,1.0,6.0,15.0,3.0,42.2,66.7,-9.0
2,2,7255854,-125823,BLAST Open Spring 2026,288898,Natus Vincere,288899,Team Vitality,0,3,...,8.0,76.9,72.0,2.0,46.0,25.0,9.0,107.8,88.2,21.0
1,1,7255854,125826,BLAST Open Spring 2026,288898,Natus Vincere,288899,Team Vitality,0,3,...,8.0,95.2,73.9,4.0,19.0,12.0,4.0,75.0,91.3,7.0


In [118]:
df.datetime

9071   2023-10-27 11:00:00
9068   2023-10-27 11:00:00
9070   2023-10-27 11:00:00
9069   2023-10-27 11:00:00
9067   2023-10-27 14:15:00
               ...        
4      2026-03-28 19:40:00
3      2026-03-28 19:40:00
2      2026-03-29 13:30:00
1      2026-03-29 13:30:00
0      2026-03-29 13:30:00
Name: datetime, Length: 9072, dtype: datetime64[ns]

In [119]:
#Drop those that have NaN
df = df.dropna()
df

,Unnamed: 0,match_id,game_id,tournament,team1_id,team1,team2_id,team2,score1_match,score2_match,...,team2_player4_assists,team2_player4_adr,team2_player4_kast,team2_player4_kddiff,team2_player5_kills,team2_player5_deaths,team2_player5_assists,team2_player5_adr,team2_player5_kast,team2_player5_kddiff
9068,9068,589049,115902,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,7.0,80.1,83.3,1.0,13.0,16.0,4.0,49.0,58.3,-3.0
9070,9070,589049,115901,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,2.0,48.0,66.7,-4.0,9.0,17.0,6.0,74.0,61.9,-8.0
9069,9069,589049,115903,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,5.0,54.0,69.6,-6.0,17.0,17.0,2.0,68.6,69.6,0.0
9067,9067,589047,-115906,Thunderpick World Championship 2023,233631,Monte,233632,Wildcard Gaming,2,0,...,6.0,56.3,45.0,-15.0,14.0,29.0,2.0,53.7,47.6,-15.0
9066,9066,589047,115904,Thunderpick World Championship 2023,233631,Monte,233632,Wildcard Gaming,2,0,...,1.0,47.3,26.7,-11.0,2.0,14.0,0.0,23.5,26.7,-12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,4,7230540,125821,BLAST Open Spring 2026,288894,PARIVISION,288895,Natus Vincere,1,2,...,5.0,85.8,80.0,2.0,23.0,15.0,7.0,94.2,75.0,8.0
3,3,7230540,125819,BLAST Open Spring 2026,288894,PARIVISION,288895,Natus Vincere,1,2,...,6.0,78.9,76.2,1.0,6.0,15.0,3.0,42.2,66.7,-9.0
2,2,7255854,-125823,BLAST Open Spring 2026,288898,Natus Vincere,288899,Team Vitality,0,3,...,8.0,76.9,72.0,2.0,46.0,25.0,9.0,107.8,88.2,21.0
1,1,7255854,125826,BLAST Open Spring 2026,288898,Natus Vincere,288899,Team Vitality,0,3,...,8.0,95.2,73.9,4.0,19.0,12.0,4.0,75.0,91.3,7.0


Drop those that have aggregated columns to fix our granularity to game-level only

In [120]:
df = df[df.is_total == False]
df

,Unnamed: 0,match_id,game_id,tournament,team1_id,team1,team2_id,team2,score1_match,score2_match,...,team2_player4_assists,team2_player4_adr,team2_player4_kast,team2_player4_kddiff,team2_player5_kills,team2_player5_deaths,team2_player5_assists,team2_player5_adr,team2_player5_kast,team2_player5_kddiff
9068,9068,589049,115902,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,7.0,80.1,83.3,1.0,13.0,16.0,4.0,49.0,58.3,-3.0
9070,9070,589049,115901,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,2.0,48.0,66.7,-4.0,9.0,17.0,6.0,74.0,61.9,-8.0
9069,9069,589049,115903,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,5.0,54.0,69.6,-6.0,17.0,17.0,2.0,68.6,69.6,0.0
9066,9066,589047,115904,Thunderpick World Championship 2023,233631,Monte,233632,Wildcard Gaming,2,0,...,1.0,47.3,26.7,-11.0,2.0,14.0,0.0,23.5,26.7,-12.0
9065,9065,589047,115906,Thunderpick World Championship 2023,233631,Monte,233632,Wildcard Gaming,2,0,...,5.0,65.3,63.2,-4.0,12.0,15.0,2.0,83.8,68.4,-3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,5,7230540,125818,BLAST Open Spring 2026,288894,PARIVISION,288895,Natus Vincere,1,2,...,7.0,96.1,91.7,14.0,20.0,15.0,0.0,69.0,79.2,5.0
4,4,7230540,125821,BLAST Open Spring 2026,288894,PARIVISION,288895,Natus Vincere,1,2,...,5.0,85.8,80.0,2.0,23.0,15.0,7.0,94.2,75.0,8.0
3,3,7230540,125819,BLAST Open Spring 2026,288894,PARIVISION,288895,Natus Vincere,1,2,...,6.0,78.9,76.2,1.0,6.0,15.0,3.0,42.2,66.7,-9.0
1,1,7255854,125826,BLAST Open Spring 2026,288898,Natus Vincere,288899,Team Vitality,0,3,...,8.0,95.2,73.9,4.0,19.0,12.0,4.0,75.0,91.3,7.0


In [121]:
df.columns

Index(['Unnamed: 0', 'match_id', 'game_id', 'tournament', 'team1_id', 'team1',
       'team2_id', 'team2', 'score1_match', 'score2_match', 'is_total',
       'bestOf', 'score1_game', 'score2_game', 'map_id', 'map_name',
       'datetime', 'team1_win', 'games_played', 'team1_player1_id',
       'team1_player2_id', 'team1_player3_id', 'team1_player4_id',
       'team1_player5_id', 'team2_player1_id', 'team2_player2_id',
       'team2_player3_id', 'team2_player4_id', 'team2_player5_id',
       'team1_player1', 'team1_player2', 'team1_player3', 'team1_player4',
       'team1_player5', 'team2_player1', 'team2_player2', 'team2_player3',
       'team2_player4', 'team2_player5', 'team1_player1_kills',
       'team1_player1_deaths', 'team1_player1_assists', 'team1_player1_adr',
       'team1_player1_kast', 'team1_player1_kddiff', 'team1_player2_kills',
       'team1_player2_deaths', 'team1_player2_assists', 'team1_player2_adr',
       'team1_player2_kast', 'team1_player2_kddiff', 'team1_player3

Let's get the average stat for the up to last 10 games this player has participated in and use that to predict if they win this game or not

In [122]:
# How many latest games do we average over
game_window = 10

#Maps player ID to dictionary of stats per game
playerDict = {}
#Maps team to dictionary of stats per game
teamDict = {}

# A fallback for if the player has no data to go off of (e.g. first time seeing them)
# Asked ChatGPT how to grab all team 1 or 2 players 1 through 5 stats
# We set our defaults initially to stats I looked up on Google which we will update iteratively
# as we collect games
default_stats = {
    "kills": 16,
    "deaths": 15,
    "assists": 4,
    "adr": 75,
    "kast": 70,
    "kddiff": 1,
}
# Number of points contributing to the global mean
n_stat = {
    "kills": 1,
    "deaths": 1,
    "assists": 1,
    "adr": 1,
    "kast": 1,
    "kddiff": 1
}
# Analogous to the above but for a team
default_team_stats = {
    "map_score": 13
}
n_stat_team = {
    "map_score": 1
}

#Asked ChatGPT for a concise way to generate the new column names
teams = [1, 2]
players = range(1, 6)
stats = ["kills", "deaths", "assists", "adr", "kast", "kddiff"]

player_stat_cols = [
    f"previous_{game_window}_game_team{t}_player{p}_average_{s}"
    for t in teams
    for p in players
    for s in stats]
additional_cols = ["team1_win", 
             "team1", 
             "team2", 
             "team1_id", 
             "team2_id", 
             "game_id", 
             "match_id", 
             "tournament",
             "bestOf",
             "map_id",
             "map_name",
             "datetime"]
player_id_cols = [f"team{t}_player{p}_id"
    for t in teams
    for p in players]
cols = []
cols.extend(player_stat_cols)
cols.extend(additional_cols)
cols.extend([f"team1_previous_{game_window}_average_map_score",
             f"team2_previous_{game_window}_average_map_score"])
cols.extend(player_id_cols)
print(cols)

#The DataFrame we are building
entries = []
for idx, row in df.iterrows():
    engineered_entry = []
    #Iterate over both teams and their 5 players
    for k in range(1,3):
        for i in range(1,6):
            #Pull previous stats
            player_i_id = row[f"team{k}_player{i}_id"]
            player_df = playerDict.get(player_i_id)
            #Give a default stat entry if we have no recorded data for this player
            fresh_record = False
            if(player_df is None):
                player_df = [default_stats]
                fresh_record = True
            latest_window = player_df[-game_window:]
            for stat in stats:
                engineered_entry.append(sum(entry[stat] for entry in latest_window) / len(latest_window))
            #Throw out the placeholder if needed
            if(fresh_record):
                player_df.pop()
            #Add this game to the rolling window
            #Asked Gemini how to convert the row series entries to correspond to stat labels
            player_df.append({s: row[f"team{k}_player{i}_{s}"] for s in stats})
            playerDict[player_i_id] = player_df
    # Add singleton info like team name, match id, etc
    for col in additional_cols:
        engineered_entry.append(row[col])
    #Add average map score for this map
    for i in range(1,3):
        #Pull info for this team
        team_df = teamDict.get(row[f"team{i}"])
        if(team_df is None):
            team_df = {}
        #Pull info for this map for this team
        map_df = team_df.get(row[f"map_id"])
        fresh_record = False
        if(map_df is None):
            map_df = [default_team_stats]
            fresh_record = True
        latest_window = map_df[-game_window:]
        engineered_entry.append(sum(entry["map_score"] for entry in latest_window) / len(latest_window))
        #Update this team's record
        if(fresh_record):
            map_df.pop()
        map_df.append({"map_score": row[f"score{i}_game"]})
        team_df[row[f"map_id"]] = map_df
        teamDict[row[f"team{i}"]] = team_df
    # Add player ids
    for k in range(1,3):
        for i in range(1,6):
            engineered_entry.append(row[f"team{k}_player{i}_id"])
    entries.append(engineered_entry)
    # Update global average map score
    for i in range(1, 3):
        default_team_stats["map_score"] = default_team_stats["map_score"] + ((row[f"score{i}_game"]) - default_team_stats["map_score"]) / (n_stat_team["map_score"] + 1)
        n_stat_team["map_score"] = n_stat_team["map_score"] + 1
    #Update the global mean after having observed this entry (use it for the future)
    for stat in stats:
        for i in range(1,3):
            for k in range(1,6):
                # Incrementally recompute the mean
                default_stats[stat] = default_stats[stat] + ((row[f"team{i}_player{k}_{stat}"] - default_stats[stat]) / (n_stat[stat]+1))
                n_stat[stat] = n_stat[stat] + 1
    
engineereddf = pd.DataFrame(entries, columns=cols)
engineereddf

['previous_10_game_team1_player1_average_kills', 'previous_10_game_team1_player1_average_deaths', 'previous_10_game_team1_player1_average_assists', 'previous_10_game_team1_player1_average_adr', 'previous_10_game_team1_player1_average_kast', 'previous_10_game_team1_player1_average_kddiff', 'previous_10_game_team1_player2_average_kills', 'previous_10_game_team1_player2_average_deaths', 'previous_10_game_team1_player2_average_assists', 'previous_10_game_team1_player2_average_adr', 'previous_10_game_team1_player2_average_kast', 'previous_10_game_team1_player2_average_kddiff', 'previous_10_game_team1_player3_average_kills', 'previous_10_game_team1_player3_average_deaths', 'previous_10_game_team1_player3_average_assists', 'previous_10_game_team1_player3_average_adr', 'previous_10_game_team1_player3_average_kast', 'previous_10_game_team1_player3_average_kddiff', 'previous_10_game_team1_player4_average_kills', 'previous_10_game_team1_player4_average_deaths', 'previous_10_game_team1_player4_ave

,previous_10_game_team1_player1_average_kills,previous_10_game_team1_player1_average_deaths,previous_10_game_team1_player1_average_assists,previous_10_game_team1_player1_average_adr,previous_10_game_team1_player1_average_kast,previous_10_game_team1_player1_average_kddiff,previous_10_game_team1_player2_average_kills,previous_10_game_team1_player2_average_deaths,previous_10_game_team1_player2_average_assists,previous_10_game_team1_player2_average_adr,...,team1_player1_id,team1_player2_id,team1_player3_id,team1_player4_id,team1_player5_id,team2_player1_id,team2_player2_id,team2_player3_id,team2_player4_id,team2_player5_id
0,16.000000,15.000000,4.000000,75.000000,70.000000,1.000000,16.000000,15.000000,4.000000,75.000000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
1,12.000000,17.000000,4.000000,59.100000,62.500000,-5.000000,13.000000,11.000000,2.000000,52.600000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
2,13.500000,16.500000,4.500000,66.250000,71.750000,-3.000000,18.500000,11.000000,4.000000,82.550000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
3,14.612903,14.645161,4.258065,69.832258,71.535484,-0.032258,14.612903,14.645161,4.258065,69.832258,...,3202.0,1420.0,3168.0,607.0,3032.0,3836.0,1886.0,2588.0,1778.0,4567.0
4,13.000000,5.000000,1.000000,81.300000,86.700000,8.000000,11.000000,6.000000,4.000000,72.100000,...,3202.0,1420.0,3168.0,607.0,3032.0,3836.0,1886.0,2588.0,1778.0,4567.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5534,15.400000,14.800000,4.900000,73.770000,77.820000,0.600000,14.200000,12.900000,3.900000,66.030000,...,2988.0,266.0,7528.0,6900.0,16113.0,1401.0,169.0,1834.0,2934.0,7201.0
5535,16.600000,15.500000,5.000000,76.570000,76.160000,1.100000,14.900000,13.400000,3.900000,66.480000,...,2988.0,266.0,7528.0,6900.0,16113.0,1401.0,169.0,1834.0,2934.0,7201.0
5536,15.400000,15.900000,4.600000,73.230000,73.160000,-0.500000,14.800000,13.600000,3.700000,66.540000,...,2988.0,266.0,7528.0,6900.0,16113.0,1401.0,169.0,1834.0,2934.0,7201.0
5537,13.300000,13.400000,4.500000,67.660000,73.790000,-0.100000,10.700000,13.700000,5.800000,57.120000,...,1401.0,169.0,1834.0,2934.0,7201.0,19.0,550.0,1549.0,1443.0,146.0


In [124]:
engineereddf.columns

Index(['previous_10_game_team1_player1_average_kills',
       'previous_10_game_team1_player1_average_deaths',
       'previous_10_game_team1_player1_average_assists',
       'previous_10_game_team1_player1_average_adr',
       'previous_10_game_team1_player1_average_kast',
       'previous_10_game_team1_player1_average_kddiff',
       'previous_10_game_team1_player2_average_kills',
       'previous_10_game_team1_player2_average_deaths',
       'previous_10_game_team1_player2_average_assists',
       'previous_10_game_team1_player2_average_adr',
       'previous_10_game_team1_player2_average_kast',
       'previous_10_game_team1_player2_average_kddiff',
       'previous_10_game_team1_player3_average_kills',
       'previous_10_game_team1_player3_average_deaths',
       'previous_10_game_team1_player3_average_assists',
       'previous_10_game_team1_player3_average_adr',
       'previous_10_game_team1_player3_average_kast',
       'previous_10_game_team1_player3_average_kddiff',
       

# We can now save our engineered dataset!

In [125]:
engineereddf.to_csv("../../data/rolling_feature_engineered_tier_1_games.csv", index=False)
print("Saved to data/rolling_feature_engineered_tier_1_games.csv")

Saved to data/rolling_feature_engineered_tier_1_games.csv


I'd like to get some stats on this new dataset as well

In [126]:
engineereddf.shape

(5539, 84)

In [127]:
engineereddf.isna().sum().sum()

np.int64(0)

No missing values in the engineered set

In [128]:
engineereddf.team1_win.value_counts()

team1_win
1    3040
0    2499
Name: count, dtype: int64

The balancing issue compared against the raw dataset is pretty much gone. There are about 500 more wins than losses, but the margin is much smaller.